# Statistical tests — is this difference real?

_Metrics & Evaluation — measuring models, data & statistics_

**A p-value tells you how surprised to be if nothing was really going on. Small p, real effect (probably).**

A hypothesis test answers one question: could this result have happened just by chance?

       You start by playing devil's advocate. You assume the boring explanation is true — that there is no real effect. This boring assumption is called the null hypothesis, written $H_0$ (read "H-naught"). For a t-test it means "the two groups have the same true average."

---

This notebook is a practice scaffold for the **Statistical tests — is this difference real?** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — scipy / statsmodels

In [ ]:
import numpy as np
from scipy import stats
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.diagnostic import acorr_ljungbox, het_breuschpagan
from statsmodels.tsa.stattools import adfuller

rng = np.random.default_rng(0)
A = rng.normal(0.0, 1.0, size=40)        # group A
B = rng.normal(0.6, 1.2, size=40)        # group B (shifted, wider spread)

# ---- compare two group averages ----------------------------------------
print("Welch t-test :", stats.ttest_ind(A, B, equal_var=False))   # unequal spread -> Welch
print("paired t-test:", stats.ttest_rel(A, B))                    # same items measured twice
print("Mann-Whitney :", stats.mannwhitneyu(A, B))                 # nonparametric (no normality)
print("Wilcoxon     :", stats.wilcoxon(A, B))                     # nonparametric paired

# ---- three or more groups ----------------------------------------------
C = rng.normal(0.3, 1.0, size=40)
print("ANOVA F-test :", stats.f_oneway(A, B, C))                  # parametric, 3+ groups
print("Kruskal-Wall :", stats.kruskal(A, B, C))                   # nonparametric, 3+ groups

# ---- counts in categories ----------------------------------------------
table = np.array([[30, 10], [18, 22]])                            # 2x2 contingency table
print("chi-square   :", stats.chi2_contingency(table)[:2])        # statistic, p-value
print("Fisher exact :", stats.fisher_exact(table))               # exact, for small counts

# ---- assumption checks --------------------------------------------------
print("Shapiro-Wilk :", stats.shapiro(A))                        # H0: data is normal
print("Levene       :", stats.levene(A, B))                      # H0: equal variances

# ---- compare two classifiers on the SAME examples (paired) -------------
# off-diagonal = cases where the two models disagree
disagree = np.array([[0, 13],     # [both wrong, A right & B wrong]
                     [27, 0]])    # [A wrong & B right, both right]
print("McNemar      :", mcnemar(disagree, exact=True))           # H0: models equally good

# ---- regression / time-series diagnostics ------------------------------
resid = rng.normal(0, 1, size=120)
print("Ljung-Box    :\n", acorr_ljungbox(resid, lags=[10]))      # H0: no autocorrelation
x = np.column_stack([np.ones(120), np.arange(120)])
print("Breusch-Pagan:", het_breuschpagan(resid, x)[:2])          # H0: constant variance
print("ADF (statn.) :", adfuller(np.cumsum(resid))[:2])          # H0: non-stationary (small p = stationary)

## Visualize the data & results

_Which breast-cancer measurements actually differ between malignant and benign tumors? A two-sample (Welch) t-test per feature, plotted as −log10(p) so taller bars = more significant._

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from scipy import stats

data = load_breast_cancer()
X, y, names = data.data, data.target, data.feature_names
# sklearn target: 0 = malignant, 1 = benign
malignant = X[y == 0]
benign    = X[y == 1]

# Welch two-sample t-test for every one of the 30 features
neglogp = {}
for i, name in enumerate(names):
    t, p = stats.ttest_ind(malignant[:, i], benign[:, i], equal_var=False)
    neglogp[name] = -np.log10(p)

# the eight features shown (a spread from most to least significant)
show = ["worst concave points", "mean radius", "mean texture",
        "worst smoothness", "mean smoothness", "mean symmetry",
        "mean fractal dimension", "symmetry error"]
for name in show:
    print(f"{name:24s} -log10(p) = {neglogp[name]:.2f}")

print("significance line -log10(0.05) =", round(-np.log10(0.05), 2))  # 1.30

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You compare model A and model B on the same 2,000-image test set. You want to claim B is genuinely better, not lucky. Which test should you use, and why not a plain two-sample t-test?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Notice the data is paired. — _Both models are scored on the exact same images, so each image gives a matched pair of correct/wrong outcomes — the runs are not independent._
- Rule out the unpaired t-test. — _A two-sample t-test treats the two runs as separate groups and ignores the pairing, throwing away information and weakening the test._
- Pick the paired test for classifiers. — _McNemar's test looks only at the images where the two models disagree (one right, one wrong) and tests whether those disagreements favor B._

**Answer:** Use McNemar's test. Because both models are scored on the same examples, the data is paired: each image is a matched outcome for A and B. McNemar's focuses on the discordant pairs — the cases where exactly one model is right — and asks whether they lean toward B more than chance allows. A plain two-sample t-test would treat the runs as independent groups, discard the pairing, and give a weaker, less honest answer.

</details>

**Problem 2.** A teammate runs t-tests on 40 features to see which differ between two customer groups, and reports the 3 features with p < 0.05 as "significant findings." What is wrong, and how do you fix it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Count the expected false alarms. — _With α = 0.05 and 40 tests, even if NO feature truly differs you expect about 40 × 0.05 = 2 features to cross p < 0.05 purely by chance._
- Name the problem. — _This is the multiple-comparisons problem (a form of p-hacking when you cherry-pick the winners): testing many things inflates the chance of a false positive somewhere._
- Apply a correction. — _Bonferroni shrinks the cutoff to 0.05/40 = 0.00125 (very strict); Benjamini–Hochberg controls the false discovery rate (FDR) for a gentler, more powerful correction._

**Answer:** The flaw is multiple comparisons: across 40 independent tests at α = 0.05 you expect roughly 2 features to look "significant" by pure chance, so 3 hits is barely above noise. Fix it with a multiple-comparison correction — Bonferroni (compare each p to 0.05/40 ≈ 0.00125) for a strict guarantee, or Benjamini–Hochberg FDR for a less conservative control of the false-discovery rate. Report only the features that survive the correction.

</details>

**Problem 3.** You have two groups of reaction times that are strongly right-skewed (a few very slow responses). You also have a giant sample (n = 500,000 per group) and a t-test returns p = 0.0003. Name two separate concerns.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Check the t-test's assumption. — _The t-test assumes roughly normal data; strong skew with heavy tails can make its p-value untrustworthy, especially the way outliers pull the mean._
- Pick a robust alternative. — _A nonparametric test (Mann–Whitney U) or a permutation test makes no normality assumption and is safer on skewed data._
- Question the giant sample. — _With n = 500,000, even a difference far too small to matter will be "statistically significant" — a tiny p says nothing about whether the effect is large._

**Answer:** Two concerns. (1) Assumption: the data is strongly skewed, so the t-test's normality assumption is shaky — switch to a Mann–Whitney U or a permutation test, which need no bell-curve assumption. (2) Practical significance: with half a million points per group, even a meaningless difference comes out p < 0.05, so the tiny p is not evidence the effect matters. Report the effect size and a confidence interval and ask whether the gap is big enough to act on.

</details>